In [ ]:
# ============================================================
# CHALLENGE 2 — Fake News Detection
# Model: roberta-base | Metric: Overall Accuracy
# ============================================================

!pip install transformers datasets scikit-learn -q

In [ ]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
# ── 1. Load Data ──────────────────────────────────────────
train = pd.read_csv("fakenews_with_labels.csv",
                    encoding='latin-1', on_bad_lines='skip')
test  = pd.read_csv("FakeNews_no_labels.csv",
                    encoding='latin-1', on_bad_lines='skip')

In [ ]:
# Fix BOM character in title column
train.columns = train.columns.str.strip().str.replace('ÿ', '', regex=False)
test.columns  = test.columns.str.strip().str.replace('ÿ', '', regex=False)

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Train shape:", train.shape)
print("Label distribution:\n", train['label'].value_counts())

In [ ]:
# ── 2. Clean Data ─────────────────────────────────────────
# Drop nulls
train = train.dropna(subset=['label']).reset_index(drop=True)
train['title'] = train['title'].fillna('')
train['text']  = train['text'].fillna('')
test['title']  = test['title'].fillna('')
test['text']   = test['text'].fillna('')

# Combine title + text + subject (max signal)
train['combined'] = (train['title'] + ' [SEP] ' +
                     train['text']  + ' [SEP] ' +
                     train['subject'].fillna(''))
test['combined']  = (test['title']  + ' [SEP] ' +
                     test['text']   + ' [SEP] ' +
                     test['subject'].fillna(''))


In [ ]:
# Truncate to 512 words max (BERT limit)
train['combined'] = train['combined'].apply(lambda x: ' '.join(x.split()[:400]))
test['combined']  = test['combined'].apply(lambda x: ' '.join(x.split()[:400]))

TEXT_COL  = 'combined'
LABEL_COL = 'label'

print("\nSample combined text:", train['combined'].iloc[0][:200])

In [ ]:
# ── 3. Encode Labels ──────────────────────────────────────
# Labels are "True" / "False" strings
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train['label_enc'] = le.fit_transform(train[LABEL_COL])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
# False=0, True=1


In [ ]:
# ── 4. Baseline — TF-IDF + Logistic Regression ───────────
print("\n--- Running Baseline ---")
vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vec.fit_transform(train[TEXT_COL])
y = train['label_enc']

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
scores = cross_val_score(lr, X, y, cv=5, scoring='accuracy')
print(f"Baseline CV Accuracy: {scores.mean():.4f}")

In [ ]:
# Save baseline submission
lr.fit(X, y)
baseline_preds_enc = lr.predict(vec.transform(test[TEXT_COL]))
baseline_preds = le.inverse_transform(baseline_preds_enc)

test_baseline = pd.read_csv("FakeNews_no_labels.csv",
                             encoding='latin-1', on_bad_lines='skip')
test_baseline.columns = test_baseline.columns.str.strip().str.replace('ÿ', '', regex=False)
test_baseline['label'] = baseline_preds
test_baseline.to_csv("FakeNews_no_labels_baseline.csv", index=False)
print("✅ Baseline submission saved")

In [ ]:
# ── 5. Main Model — RoBERTa ───────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

MODEL = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True,
                     padding="max_length", max_length=256)

In [ ]:
# Build HF Dataset
train_ds = Dataset.from_pandas(
    train[[TEXT_COL, 'label_enc']].rename(columns={'label_enc': 'label'})
)
train_ds = train_ds.map(tokenize, batched=True)
split = train_ds.train_test_split(test_size=0.1, seed=42)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="./c2_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    seed=42,
    logging_steps=100,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
)
trainer.train()

In [ ]:
#  6. Evaluation
val_preds = trainer.predict(split['test'])
val_labels_pred = np.argmax(val_preds.predictions, axis=1)
val_labels_true = split['test']['label']

print("\n" + "="*50)
print("VALIDATION RESULTS")
print("="*50)
print(classification_report(
    val_labels_true, val_labels_pred,
    target_names=[str(c) for c in le.classes_]
))
print(f"Accuracy: {accuracy_score(val_labels_true, val_labels_pred):.4f}")
print("="*50)

In [ ]:
# ── 7. Predict & Save Final Submission ───────────────────
test_ds = Dataset.from_pandas(test[[TEXT_COL]])
test_ds = test_ds.map(tokenize, batched=True)

final_preds = trainer.predict(test_ds)
final_enc   = np.argmax(final_preds.predictions, axis=1)
final_labels = le.inverse_transform(final_enc)  # "True" / "False"

In [ ]:
# Fill label column — keep original structure
test_final = pd.read_csv("FakeNews_no_labels.csv",
                          encoding='latin-1', on_bad_lines='skip')
test_final.columns = test_final.columns.str.strip().str.replace('ÿ', '', regex=False)
test_final['label'] = final_labels


In [ ]:
# Verify
print("\nPredicted label values:", test_final['label'].value_counts().to_dict())
print("Expected: True / False")
print("Null labels:", test_final['label'].isnull().sum())

test_final.to_csv("FakeNews_no_labels.csv", index=False)


In [ ]:
from google.colab import files
files.download("FakeNews_no_labels.csv")

In [ ]:
print("✅ Final submission saved — FakeNews_no_labels.csv")